# Tools and Permissions — Claude Managed Agents (CLI)

The same flow as [`03-tools-and-permissions/`](../03-tools-and-permissions/), driven with the **`ant`** CLI. In modules 01 and 02 the agent ran its tools on its own; here **your client takes part in the turn**. Two kinds of tool call pause the agent until you answer:

| Pause | The agent emits | You answer with |
|---|---|---|
| A **custom tool** — a tool you define and your code runs | `agent.custom_tool_use` | `user.custom_tool_result` |
| A **built-in tool** whose permission policy asks first | `agent.tool_use` with `evaluated_permission: "ask"` | `user.tool_confirmation` (`allow` / `deny`) |

The CLI angle: the whole tool configuration — toolset overrides, permission policies, the custom tool's schema — lives in the agent's **frontmatter**, reviewable in a pull request. Answering calls is data-plane work, so it happens in a Python loop that reads `ant`'s event stream and replies with `ant ... send`.

> Prerequisite: the `ant` CLI (>= 1.40) installed and authenticated — see [`01-basics-cli/`](../01-basics-cli/).

## 1. Setup

Same as the previous CLI modules: load the root `.env` for `ant`, and keep its output plain.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv

# `ant` is a separate process from this notebook, so a bit of setup:
#  - it doesn't read the root .env — load the API key into os.environ, which
#    every `!ant` call below inherits;
#  - Jupyter sets FORCE_COLOR=1, which makes ant wrap its JSON output in
#    ANSI color codes and break parsing — clear it so the output stays plain;
#  - `!` runs commands in a pseudo-terminal, so ant probes it with escape
#    sequences that leak into the cell output — TERM=dumb turns that off.
load_dotenv(find_dotenv())
os.environ.pop("FORCE_COLOR", None)
os.environ.pop("CLICOLOR_FORCE", None)
os.environ["NO_COLOR"] = "1"
os.environ["TERM"] = "dumb"
assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY not found — add it to the root .env or run `ant auth login`."
)
print("Credential loaded, color disabled — ant is ready")

## 2. Create an Environment

Declared in `environment.yaml`, reconciled with `ant apply`. Re-running the cell is a no-op thanks to the lockfile.

In [ ]:
%%writefile environment.yaml
# yaml-language-server: $schema=https://platform.claude.com/schemas/ant/beta/environment.json
name: tools-permissions-env-cli
config:
  type: cloud
  networking:
    type: unrestricted

In [ ]:
import json

!ant apply environment.yaml --yes --lock-file claude-lock.json

lock = json.load(open("claude-lock.json"))
env_id = lock["resources"]["./environment.yaml"]["id"]
print("Environment ID:", env_id)

## 3. Declare the Agent's Tools

Everything about the tools goes in the frontmatter's `tools` list:

**The built-in toolset, tuned with `configs`.** Each entry names one tool (`bash`, `read`, `write`, `edit`, `glob`, `grep`, `web_fetch`, `web_search`) and overrides it:

- `enabled: false` removes the tool. Our inventory agent has no business on the web.
- `permission_policy` controls *when* it runs:

| Policy | Behavior |
|---|---|
| `always_allow` | Runs with no confirmation. **Default for the built-in toolset.** |
| `always_ask` | The session pauses until your client sends `allow` or `deny`. |
| `auto` | The server evaluates each call: runs it, denies it, or pauses for your approval. |

We put `bash` under `always_ask`: every shell command waits for us.

**A custom tool**, `lookup_stock`: a name, a description, and a JSON Schema for its input. The agent can call it, but **your code runs it** — Claude only sees the result you send back. Custom tools are not governed by permission policies; your code decides whether to run them.

In [ ]:
%%writefile agent.md
---
name: Inventory Agent (CLI)
model: claude-opus-5-5
tools:
  - type: agent_toolset_20260401
    configs:
      - name: web_search
        enabled: false
      - name: web_fetch
        enabled: false
      - name: bash
        permission_policy:
          type: always_ask
  - type: custom
    name: lookup_stock
    description: >-
      Look up how many units of a product are in the warehouse.
      Use this for every stock question; never guess stock levels.
      Pass the SKU exactly as the user wrote it, for example A-100.
      Returns the unit count, or an error if the SKU is unknown.
    input_schema:
      type: object
      properties:
        sku:
          type: string
          description: Product SKU, for example A-100
      required:
        - sku
---

You are an inventory assistant. Use the lookup_stock tool for stock levels.
Keep your answers short.

In [ ]:
!ant apply agent.md --yes --lock-file claude-lock.json

lock = json.load(open("claude-lock.json"))
agent_id = lock["resources"]["./agent.md"]["id"]
agent_version = int(lock["resources"]["./agent.md"]["version"])
print(f"Agent ID : {agent_id}  (v{agent_version})")

## 4. Implement the Custom Tool and an Approval Policy

The tool's implementation lives in *your* system — here, a Python dict standing in for a warehouse database.

When a `bash` call pauses, someone has to decide. In a real app that's often a person; to keep the notebook runnable top to bottom, `approve()` is a rule: shell commands are fine, deleting files is not. A denial carries a `deny_message` that the agent receives as the tool's result.

> **Approving by hand from the terminal:** `ant beta:sessions connect --session-id <id>` follows a session interactively and lets you allow or deny each waiting call yourself.

In [ ]:
STOCK = {"A-100": 42, "B-200": 0, "C-300": 7}


def lookup_stock(sku: str) -> tuple[str, bool]:
    """Run the tool in our system. Returns (result text, is_error)."""
    if sku not in STOCK:
        return json.dumps({"error": f"Unknown SKU: {sku}"}), True
    return json.dumps({"sku": sku, "units": STOCK[sku]}), False


CUSTOM_TOOLS = {"lookup_stock": lookup_stock}


def approve(command: str) -> tuple[str, str | None]:
    """Decide on a bash call. Returns ("allow" | "deny", deny_message)."""
    if "rm " in command:
        return "deny", "Deleting files is not allowed in this workspace."
    return "allow", None


print(lookup_stock("A-100"), lookup_stock("Z-999"))

## 5. The Event Loop: Answer Each Call as It Arrives

The loop from module 02 grows two branches, one per kind of pause. Each answer is one `ant beta:sessions:events send`, carrying the event's `id`:

- `agent.custom_tool_use` → run the tool, send `user.custom_tool_result` (`custom_tool_use_id`)
- `agent.tool_use` with `evaluated_permission == "ask"` → decide, send `user.tool_confirmation` (`tool_use_id`)

And it reads the idle's `stop_reason` before stopping:

| `stop_reason.type` | Meaning | What the loop does |
|---|---|---|
| `end_turn` | The agent finished its turn | Stop reading |
| `requires_action` | Calls are waiting on you (`stop_reason.event_ids`) | **Keep reading** — we answer each call as its event arrives |
| `budget_reached` | The session hit its spending cap (module 05) | Stop reading |
| `retries_exhausted`, `refusal` | The session can't continue this turn | Stop reading |

We answer **on arrival**, not on the `requires_action` idle: once a session runs several agents (modules 06–07), it can stay `running` while one waits on you, and that idle never comes.

As in the earlier CLI modules, the stream is read with `--format json`: each event arrives as an indented JSON object that closes with `}` at column 0.

In [ ]:
import subprocess


def send(session_id: str, *events: dict) -> None:
    """Send events to the session with `ant beta:sessions:events send`."""
    subprocess.run(
        ["ant", "beta:sessions:events", "send", "--session-id", session_id],
        input=json.dumps({"events": list(events)}),
        stdout=subprocess.DEVNULL, text=True, check=True,
    )


def events(stream):
    """Yield each event from `ant ... stream --format json` as a dict."""
    buf = []
    for line in stream.stdout:
        buf.append(line)
        if line.rstrip("\n") == "}":   # a top-level object closes at column 0
            yield json.loads("".join(buf))
            buf = []


def run_turn(session_id: str, text: str) -> None:
    """Send one user message, answer every tool call, and stream until the turn ends."""
    # Open the stream FIRST, then send.
    stream = subprocess.Popen(
        ["ant", "beta:sessions:events", "stream", "--session-id", session_id, "--format", "json"],
        stdout=subprocess.PIPE, text=True, bufsize=1,
    )
    send(session_id, {"type": "user.message", "content": [{"type": "text", "text": text}]})

    for event in events(stream):
        match event["type"]:
            case "agent.message":
                for block in event["content"]:
                    if block["type"] == "text":
                        print(f"Agent: {block['text']}")

            case "agent.custom_tool_use":
                # A custom tool: run it in our system and send the result back.
                result, is_error = CUSTOM_TOOLS[event["name"]](**event["input"])
                print(f"[custom tool] {event['name']}({event['input']}) -> {result}")
                send(session_id, {
                    "type": "user.custom_tool_result",
                    "custom_tool_use_id": event["id"],
                    "content": [{"type": "text", "text": result}],
                    "is_error": is_error,
                })

            case "agent.tool_use":
                # A built-in tool. Under always_ask it waits for our confirmation.
                detail = event["input"].get("command") or event["input"].get("file_path") or ""
                policy = (event.get("evaluation") or {}).get("type", "-")
                print(f"[{event['name']}] {detail}  (permission: {event.get('evaluated_permission')}, policy: {policy})")
                if event.get("evaluated_permission") == "ask":
                    decision, deny_message = approve(detail)
                    confirmation = {"type": "user.tool_confirmation", "tool_use_id": event["id"], "result": decision}
                    if deny_message:
                        confirmation["deny_message"] = deny_message
                    send(session_id, confirmation)
                    print(f"    -> {decision}" + (f": {deny_message}" if deny_message else ""))

            case "session.status_idle":
                reason = event["stop_reason"]["type"]
                if reason == "requires_action":
                    # We answer each call as its event arrives, so keep reading.
                    print(f"[idle: requires_action, waiting on {len(event['stop_reason']['event_ids'])} call(s)]")
                    continue
                print(f"[idle: {reason}]")
                break

            case "session.status_terminated":
                print("[terminated]")
                break
    stream.terminate()


print("run_turn() ready")

## 6. Run It

Three stock lookups (one SKU doesn't exist), a `bash` command we'll allow, and a delete our policy will deny.

In [ ]:
agent_ref = json.dumps({"type": "agent", "id": agent_id, "version": agent_version})

_session = !ant beta:sessions create --agent '{agent_ref}' --environment-id {env_id} --title "Tools and permissions CLI" --transform id -r  # type: ignore
session_id = _session[0]
print("Session ID:", session_id)

In [ ]:
run_turn(
    session_id,
    "Check stock for A-100, B-200 and Z-999. Then use bash to write the results to "
    "/workspace/stock.csv (columns sku,units) and show it with cat. "
    "Finally, use bash to delete /workspace/stock.csv.",
)

### Reading the trace

- **Custom tool calls** are answered right away; `Z-999` comes back with `is_error: true`, and the agent reports it instead of inventing a number.
- **`requires_action` idles** often show up *after* we already answered, with a shrinking count — the idle and our answers cross in flight, so the loop keeps reading.
- **Allowed `bash`** runs once we send `allow`; the **denied** delete never runs, and the agent receives our `deny_message` as the tool result.
- **`end_turn`** — only now is the turn over.

## 7. Switch `bash` to `auto` — an Update Through `ant apply`

With `auto`, the server evaluates each call: it **runs** calls it judges safe, **denies** high-risk ones (the agent gets an error result and your client can't override it), and **pauses for you** when it can't decide. The trace's `policy: auto` and `permission:` show which happened.

This is also how you **update** a resource from the CLI: edit the file and apply again. `ant apply` sees the agent is already in the lockfile, plans an *update* instead of a create, and the agent gets a new version. The session above keeps the version it pinned.

In [ ]:
%%writefile agent.md
---
name: Inventory Agent (CLI)
model: claude-opus-5-5
tools:
  - type: agent_toolset_20260401
    configs:
      - name: web_search
        enabled: false
      - name: web_fetch
        enabled: false
      - name: bash
        permission_policy:
          type: auto
  - type: custom
    name: lookup_stock
    description: >-
      Look up how many units of a product are in the warehouse.
      Use this for every stock question; never guess stock levels.
      Pass the SKU exactly as the user wrote it, for example A-100.
      Returns the unit count, or an error if the SKU is unknown.
    input_schema:
      type: object
      properties:
        sku:
          type: string
          description: Product SKU, for example A-100
      required:
        - sku
---

You are an inventory assistant. Use the lookup_stock tool for stock levels.
Keep your answers short.

In [ ]:
!ant apply agent.md --yes --lock-file claude-lock.json

lock = json.load(open("claude-lock.json"))
agent_id = lock["resources"]["./agent.md"]["id"]
agent_version = int(lock["resources"]["./agent.md"]["version"])
print(f"Agent ID : {agent_id}  (v{agent_version})")

In [ ]:
agent_ref = json.dumps({"type": "agent", "id": agent_id, "version": agent_version})

_session = !ant beta:sessions create --agent '{agent_ref}' --environment-id {env_id} --title "Tools and permissions CLI (auto)" --transform id -r  # type: ignore
auto_session_id = _session[0]
print("Session ID:", auto_session_id)

run_turn(auto_session_id, "Use bash to show the current date and the free disk space of /workspace.")

A harmless command like this one typically comes back `permission: allow` under `policy: auto`, and runs with no confirmation from us.

> **`auto` is not a human checkpoint.** A call the server judges safe runs before anyone sees it. Keep `always_ask` on any tool whose calls a person must review.

## Summary

```
agent.md frontmatter   →  toolset configs (enabled, permission_policy) + custom tool schema
ant apply agent.md     →  create, or update to a new version after an edit
agent.custom_tool_use  →  run it, ant ... send user.custom_tool_result   (on arrival)
agent.tool_use "ask"   →  decide, ant ... send user.tool_confirmation    (on arrival)
session.status_idle    →  requires_action: keep reading
                          end_turn / budget_reached / ...: the turn is over
```

### CLI vs SDK, same lesson

| | CLI (this module) | SDK ([`03-tools-and-permissions/`](../03-tools-and-permissions/)) |
|---|---|---|
| Tool configuration | `agent.md` frontmatter | `tools=[...]` in `agents.create()` |
| Update the agent | edit the file, `ant apply` again | `agents.update(id, version=..., tools=...)` |
| Answer a call | `ant beta:sessions:events send` | `client.beta.sessions.events.send()` |
| Approve by hand | `ant beta:sessions connect` | your own UI |

### Next steps
- **04-files** — mount files into the sandbox and download what the agent produces
- **05-session-control** — steer, interrupt, reconnect, and cap spend with a session budget
- **06-subagents** — delegate to other agents in session threads
- **07-dynamic-workflows** — let the agent write a workflow that runs many agents in parallel